### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="airfoil_self_noise",
    dataset_year="2014",
    domain_str="physics & astronomy",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5VW2C",
    download_description="""
mkdir -p local-data-warehouse/airfoil_self_noise/ \
&& wget -P local-data-warehouse/airfoil_self_noise/ https://archive.ics.uci.edu/static/public/291/airfoil+self+noise.zip \
&& unzip local-data-warehouse/airfoil_self_noise/airfoil+self+noise.zip -d local-data-warehouse/airfoil_self_noise/
""",
    # References
    academic_reference_bibtex="""@techreport{brooks1989airfoil,
  title={Airfoil self-noise and prediction},
  author={Brooks, Thomas F and Pope, D Stuart and Marcolini, Michael A},
  year={1989}
}
""",
    academic_reference_bibtex_key="brooks1989airfoil",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""N/A""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="scaled-sound-pressure",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/airfoil_self_noise.dat", header=None, sep="\s+")

target_feature = "scaled-sound-pressure"
df.columns = [
    "frequency",
    "attack-angle",
    "chord-length",
    "free-stream-velocity",
    "suction-side-displacement-thickness",
    target_feature,
]

cat_features = [
    "attack-angle",
]

# Data is ordered by chord-length, thus dist shift for original order
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,503
Columns: 6
Use sampling: False (sample size: 1,503)
Get row duplicates (staged, merged)...
Using top-5 columns for initial filtering: ['suction-side-displacement-thickness', 'attack-angle', 'frequency', 'chord-length', 'free-stream-velocity']
Rows remaining as candidates after top-5 filter: 0 (of 1,503)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,frequency,attack-angle,chord-length,free-stream-velocity,suction-side-displacement-thickness,scaled-sound-pressure
0,400,0.0,0.3048,31.7,0.003313,125.045
1,1250,12.3,0.1016,31.7,0.041876,118.767
2,2500,4.0,0.3048,39.6,0.005796,120.233
3,4000,0.0,0.0254,31.7,0.000439,137.047
4,5000,0.0,0.0508,55.5,0.000762,134.556


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,attack-angle,category,0.0,0.0,27.0,"0.0, 4.0, 15.4, 9.9, 12.3, 7.3, 17.4, 3.0, 2.0, 9.5"
1,chord-length,float64,0.0,0.0,6.0,"0.0254, 0.1524, 0.2286, 0.1016, 0.0508, 0.3048"
2,free-stream-velocity,float64,0.0,0.0,4.0,"39.6, 71.3, 31.7, 55.5"
3,suction-side-displacement-thickness,float64,0.0,0.0,105.0,"0.0053, 0.0031, 0.0033, 0.005, 0.0091, 0.004, 0.0161, 0.0122, 0.013, 0.0264"
4,scaled-sound-pressure,float64,0.0,0.0,1456.0,"129.395, 126.54, 127.315, 125.586, 133.79, 134.058, 133.04, 130.307, 126.195, 125.201"
5,frequency,int64,0.0,0.0,21.0,"2000, 2500, 1600, 3150, 4000, 1250, 1000, 800, 5000, 6300"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
frequency,1503.0,2886.380572,3152.573137,200.000000,20000.000000
chord-length,1503.0,0.136548,0.093541,0.025400,0.304800
free-stream-velocity,1503.0,50.860745,15.572784,31.700000,71.300000
suction-side-displacement-thickness,1503.0,0.011140,0.013150,0.000401,0.058411
scaled-sound-pressure,1503.0,124.835943,6.898657,103.380000,140.987000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column       rank                    
attack-angle 1      0.0    329  21.89
             2      4.0     93   6.19
             3     15.4     65   4.32
             4      9.9     64   4.26
             5     12.3     64   4.26

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.419,-0.548,47.591,0.003,log,26708.0,6.049244e+14,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to airfoil_self_noise/019d5a0c-3004-72c3-b77b-bc348a86d057
019d5a0c-3004-72c3-b77b-bc348a86d057
6c79451403f65fbbea152a878f96ae30c9a051a5dbce73a5ee7c204a675cab89
